In [39]:
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

In [29]:
with open('szekspir.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [30]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [31]:
# Dividing data into train and validation splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train data, 10% validation data
train_data = data[:n]
val_data = data[n:]

In [28]:
# hyperparameters
batch_size = 32
block_size = 8 # maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 192
n_head = 6
n_layer = 6
dropout = 0.2

In [40]:
# generate a batch of data
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [33]:
# estimating loss
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [41]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # Ścieżka residualna atencji
        x = x + self.ffwd(self.ln2(x))  # ❗Brakująca ścieżka residualna feed-forward
        return x 
        
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # Ograniczenie długości kontekstu
            logits, _ = self(idx_cond)  # Obliczanie predykcji
            logits = logits[:, -1, :]  # Pobranie ostatniego tokena
            probs = F.softmax(logits, dim=-1)  # Przekształcenie na prawdopodobieństwa
            idx_next = torch.multinomial(probs, num_samples=1)  # Losowanie kolejnego tokena
            idx = torch.cat((idx, idx_next), dim=1)  # Dopisanie do sekwencji
        return idx

In [35]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

2.692673 M parameters


In [36]:
# Tworzenie optymalizatora
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # Co jakiś czas sprawdzamy straty na zbiorach treningowych i walidacyjnych
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Pobranie batcha
    xb, yb = get_batch('train')

    # Obliczenie straty
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.2443, val loss 4.2500
step 500: train loss 2.3150, val loss 2.3066
step 1000: train loss 2.1765, val loss 2.1792
step 1500: train loss 2.0943, val loss 2.1388
step 2000: train loss 2.0668, val loss 2.1120
step 2500: train loss 2.0091, val loss 2.0637
step 3000: train loss 1.9877, val loss 2.0728
step 3500: train loss 1.9609, val loss 2.0719
step 4000: train loss 1.9350, val loss 2.0395
step 4500: train loss 1.9039, val loss 2.0277
step 4999: train loss 1.9022, val loss 2.0083


In [38]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


Paoiln:
Dikess didnes you of heerch ther blook,
Hever wind a would what,
not
Which, peartult ofe theav,
And I have se be pan'st,
And with roply
in of muttr'dsing of tward I paring,
Holhee you shopp nour one cercuest muship like of atvan thy rive of luts one fear'st; law hour, splain, I ghilg,
Ay, and and of speak you uncrided as into mast, o' that to damon a this is lanks;
And amand hour parth,
Your stemeend of eyough of carbiall with whing be than self,
I am Jonoud
May, fort lifforne to Hous?

PORVENS:
Mathee, a foed
To him coled with hou?

Comer:
Would buth I wordes
And can otil dest how, so
Ba fires
Be purse your ill whe so pecome, buttle it know like sube myself thour teasur's up onpy forts
By she:
For?

RIVRLOMENES:
whe knowns buity, strow be tair alisnd your earr'st lone.

GLMORDUK:
As Lorst I seshad for o'nd in you ding as banounat commanstg to entir so whell and
-take a the shall I lord sen yourgurs,
I'll neat whid is to me sher, and to mook these pasl antage?

FlORK:
How a ma